# 02 - QAOA Execution

This notebook demonstrates how to implement and execute the Quantum Approximate Optimization Algorithm (QAOA) for Max-Cut.

## Table of Contents
1. [QAOA Circuit Overview](#overview)
2. [Building QAOA Circuits](#circuit)
3. [Classical Optimization Loop](#optimization)
4. [Execution with Qiskit](#execution)
5. [Energy Landscape Analysis](#landscape)

<a id='overview'></a>
## 1. QAOA Circuit Overview

### Algorithm Overview

QAOA combines:
1. **Quantum Circuit**: Prepares a parameterized trial state
2. **Classical Optimizer**: Searches over the variational parameters

### Circuit Structure

For $p$ layers,

$$|\psi(\boldsymbol{\gamma}, \boldsymbol{\beta})\rangle = U_M(\beta_p) U_C(\gamma_p) \cdots U_M(\beta_1) U_C(\gamma_1) |+\rangle^{\otimes n}.$$

Where:
- $U_C(\gamma) = e^{-i\gamma H_{\mathrm{int}}}$ uses the ZZ interaction term only
- $U_M(\beta) = e^{-i\beta \sum_i X_i}$ is the standard mixer
- The Max-Cut objective is recovered afterwards as $\langle C \rangle = \text{offset} + \langle H_{\mathrm{int}} \rangle$


<a id='circuit'></a>
## 2. Building QAOA Circuits

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

from src.graph_generator import GraphGenerator
from src.hamiltonian_builder import HamiltonianBuilder
from src.qaoa_circuit import QAOACircuitBuilder

# Generate test graph
gen = GraphGenerator(seed=42)
graph = gen.generate_d_regular_graph(n_nodes=8, degree=3, seed=42)

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
# Build Hamiltonian
builder = HamiltonianBuilder()
hamiltonian, offset = builder.build_maxcut_hamiltonian(graph)

print(f"Hamiltonian: {len(hamiltonian)} terms")

# Create QAOA circuit builder
n_qubits = graph.number_of_nodes()
p = 2  # Number of layers

circuit_builder = QAOACircuitBuilder(n_qubits=n_qubits, p=p)

# Build the circuit
qaoa_circuit = circuit_builder.build_qaoa_circuit(hamiltonian)

print(f"\nQAOA Circuit:")
print(f"  Qubits: {qaoa_circuit.num_qubits}")
print(f"  Depth: {qaoa_circuit.depth()}")
print(f"  Parameters: {qaoa_circuit.num_parameters}")

In [ ]:
# Visualize the circuit
plt.figure(figsize=(14, 6))
qaoa_circuit.draw(output='mpl', style='iqp', fold=40)
plt.title(f'QAOA Circuit (p={p} layers, {n_qubits} qubits)')
plt.show()

### Circuit Explanation

The circuit consists of:
1. **H gates**: Initialize the uniform superposition $|+\rangle^{\otimes n}$
2. **RZZ gates**: Implement the interaction evolution $e^{+i \gamma w_{ij} Z_i Z_j / 2}$ via `RZZ(-\gamma w_{ij})`
3. **RX gates**: Implement the mixer unitary $e^{-i \beta X_i}$

The constant term $\sum w_{ij}/2$ never appears in the circuit because it contributes only a global phase. It is added back when evaluating the Max-Cut objective.


<a id='optimization'></a>
## 3. Classical Optimization Loop

In [ ]:
from src.qaoa_optimizer import MaxCutQAOAProblem, QAOAOptimizer
from src.runtime_executor import RuntimeExecutor

problem = MaxCutQAOAProblem(
    graph=graph,
    p=1,
    executor=RuntimeExecutor(mode='noisy_simulator', backend_name='ibm_brisbane', shots=512, seed=42, simulate_noise=True),
    seed=42,
    analysis_shots=512,
    analysis_mode='same_backend',
)

objective_function = problem.objective_function

print("Backend-aware objective function defined")


In [ ]:
# Run optimization
optimizer = QAOAOptimizer(
    p=1,
    optimizer_type='SPSA',
    maxiter=12,
    seed=42
)

result = optimizer.optimize(
    objective_function=objective_function,
    n_qubits=n_qubits,
    graph=graph,
    solution_decoder=problem.decode_solution,
)

print(f"\nOptimization Results:")
print(f"  Optimal parameters: gamma={result.optimal_params[0]:.4f}, beta={result.optimal_params[1]:.4f}")
print(f"  Minimized objective: {result.optimal_value:.4f}")
print(f"  Expected cut value: {result.cut_value:.4f}")
print(f"  Representative sampled bitstring: {result.solution_bitstring}")
print(f"  Representative sampled cut: {result.sampled_cut_value}")
print(f"  Most likely bitstring: {result.most_likely_bitstring}")
print(f"  Most likely probability: {result.bitstring_probability}")
print(f"  Function evaluations: {result.n_evaluations}")
print(f"  Converged: {result.converged}")


In [ ]:
# Plot convergence
if result.history:
    from src.visualization import Visualizer
    
    viz = Visualizer()
    
    iterations, values = optimizer.get_convergence_plot_data(result)
    
    plt.figure(figsize=(10, 5))
    plt.plot(iterations, values, 'o-', color='orange')
    plt.xlabel('Iteration')
    plt.ylabel('Minimization Objective')
    plt.title('QAOA Optimization Convergence')
    plt.grid(True, alpha=0.3)
    plt.show()


<a id='execution'></a>
## 4. Execution with Qiskit Runtime

In [ ]:
# Execute using Aer simulator
from src.runtime_executor import RuntimeExecutor

# Create executor
executor = RuntimeExecutor(
    mode='local',
    shots=1024
)

# Get backend info
backend_info = executor.get_backend_info()
print(f"Backend: {backend_info}")

In [ ]:
# Re-run the optimized circuit through the execution layer
exec_result = problem.analysis_executor.execute_circuit(
    problem.build_circuit(result.optimal_params),
    problem.hamiltonian,
    offset=problem.offset,
)

print(f"Execution Result:")
print(f"  Interaction expectation: {exec_result.expectation_value:.4f}")
print(f"  Max-Cut objective: {exec_result.objective_value:.4f}")
print(f"  Sampled bitstring: {exec_result.sampled_bitstring}")
print(f"  Circuit depth: {exec_result.circuit_depth}")


<a id='landscape'></a>
## 5. Energy Landscape Analysis

In [ ]:
# Compute energy landscape
gamma_range = np.linspace(0, np.pi, 30)
beta_range = np.linspace(0, np.pi, 30)

gamma_grid, beta_grid = np.meshgrid(gamma_range, beta_range)
cost_grid = np.zeros_like(gamma_grid)

print("Computing energy landscape...")
for i in range(len(gamma_range)):
    for j in range(len(beta_range)):
        params = np.array([gamma_grid[i, j], beta_grid[i, j]])
        cost_grid[i, j] = objective_function(params)
    
    if (i + 1) % 10 == 0:
        print(f"  Completed {i+1}/{len(gamma_range)} rows")

print("Energy landscape computed!")

In [ ]:
# Plot energy landscape
plt.figure(figsize=(10, 8))

contour = plt.contourf(gamma_grid, beta_grid, cost_grid, levels=30, cmap='viridis')
plt.colorbar(contour, label='Minimization Objective')

# Mark optimal point
plt.plot(gamma_opt, beta_opt, 'r*', markersize=15, label=f'Optimal: gamma={gamma_opt:.3f}, beta={beta_opt:.3f}')

plt.xlabel(r'$\gamma$ (gamma)')
plt.ylabel(r'$\beta$ (beta)')
plt.title('QAOA Objective Landscape')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Summary

In this notebook, we covered:

1. **Circuit Construction**: The QAOA circuit applies weighted `RZZ(-gamma w_ij)` cost gates and `RX(2 beta)` mixer gates
2. **Objective Evaluation**: The Max-Cut expectation is `offset + <interaction term>`
3. **Optimization**: The classical optimizer minimizes the negative expected cut objective through the execution layer
4. **Decoded Outputs**: A representative sampled bitstring is tracked separately from the expected objective value
5. **Landscape Analysis**: The plotted surface is the minimized objective over $(gamma, beta)$
